In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:95%;}

div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li{font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:40px;}
</style>
"""))

# 1. knowlege base 구성을 위한 데이터 생성

In [9]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter # character 단위로 짜름
loader = Docx2txtLoader('tax_docx/소득세법_with_table.docx')
text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1500, #문서를 쪼갤 떄 1500글자씩 chunking
        chunk_overlap=20, #200문자씩 overlap 하여 chunking 작업
        #separators=["\n\n","\n"," "] -> 기본값   : 다음의 순서대로 chunking 시도
)
documents = loader.load_and_split(text_splitter=text_splitter)
print('chunk 갯수: ', len(documents))

chunk 갯수:  180


In [11]:
documents[43]

Document(metadata={'source': 'tax_docx/소득세법_with_table.docx'}, page_content='제54조의2(공동사업에 대한 소득공제 등 특례) 제51조의3 또는 「조세특례제한법」에 따른 소득공제를 적용하거나 제59조의3에 따른 세액공제를 적용하는 경우 제43조제3항에 따라 소득금액이 주된 공동사업자의 소득금액에 합산과세되는 특수관계인이 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액이 있으면 주된 공동사업자의 소득에 합산과세되는 소득금액의 한도에서 주된 공동사업자가 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액으로 보아 주된 공동사업자의 합산과세되는 종합소득금액 또는 종합소득산출세액을 계산할 때에 소득공제 또는 세액공제를 받을 수 있다. <개정 2012. 1. 1., 2014. 1. 1.>\n\n[전문개정 2009. 12. 31.]\n\n[제목개정 2014. 1. 1.]\n\n\n\n제4절 세액의 계산 <개정 2009. 12. 31.>\n\n\n\n제1관 세율 <개정 2009. 12. 31.>\n\n\n\n제55조(세율) ①거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액(이하 “종합소득산출세액”이라 한다)을 그 세액으로 한다. <개정 2014. 1. 1., 2016. 12. 20., 2017. 12. 19., 2020. 12. 29., 2022. 12. 31.>\n\n\n\n\n\n\n\n\n\n종합소득 과세표준\n\n세율\n\n1,400만원 이하\n\n과세표준의 6퍼센트\n\n1,400만원 초과 5,000만원 이하\n\n84만원 + (1,400만원을 초과하는 금액의 15퍼센트)\n\n5,000만원 초과 8,800만원 이하\n\n624만원 + (5,000만원을 초과하는 금액의 24퍼센트)\n\n8,800만원 초과 1억 5천만원 이하\n\n1,536만원 + (8,800만원을 초과하는 금액의 35퍼센트)\n\n1억5천만원 초과 3억원 이하\n\n3,706만원 + (1억5천만원을 초과하는 금액의

In [14]:
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
load_dotenv(overide=True) # 커널을 시작하지 않고 .env 덮어쓰기
embedding = UpstageEmbeddings(
                    model ='embedding-query'
)


In [16]:
len(embedding.embed_query("소득세법"))

4096

In [28]:
# pinecone의 index 생성하기
import time
from pinecone import Pinecone, ServerlessSpec
# pinecone 객체 생성
pc=Pinecone()  # api_key=os.getenv('PINECONE_API_KEY')

# 1. 인덱스 설정
index_name = 'tax-index-table'
dimension = len(embedding.embed_query("차원확인용"))

# 2. 인덱스 생성(해당인덱스가 없는경우)
existing_indexes = [index.name for index in pc.list_indexes()]
#print(index_name in existing_indexes)
if index_name not in existing_indexes:
    pc.create_index(
        name=index_name,
        dimension=dimension,
        metric='cosine', #유사도 측정방식
        spec=ServerlessSpec(
            cloud='aws',    #무료 플랜은 aws
            region='us-east-1' # 무료 플랜은 us-east-1만 가능
        )
    )
    #인덱스가 준비 될 때까지 대기
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(1) #1초 대기
    print(f'인덱스{index_name}를 생성해였습니다')    
else:
    print(f'인덱스 {index_name}이 이미 존재합니다')

인덱스 tax-index-table이 이미 존재합니다


In [29]:
[index.name for index in pc.list_indexes()]

['tax-index', 'tax-index-table', 'tax-index-openai']

In [30]:
index = pc.Index(index_name)
print(index.describe_index_stats())

C:\Users\mbc\anaconda3\envs\llm\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'dimension': 4096,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}


In [32]:
%%time
# 4. 문서를 임베딩해서 클라우드 vector DB(index)에 저장
# 처음 한번
#from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone
pc=Pinecone()
# database = PineconeVectorStore.from_documents(
#     documents=documents,
#     embedding=embedding,
#     index_name=index_name
# )
# 업로드한 벡터DB를 load할때 
index_name='tax-index-table'
database =PineconeVectorStore(
    embedding=embedding,
    index_name = index_name
)

CPU times: total: 0 ns
Wall time: 0 ns


# 2. 답변 생성 전 retrieval 확인


In [33]:
query= "연봉이 5천만원인 직장인의 소득세는 얼마인가요?"
retrieved_docs = database.similarity_search(query,k=3) # 기본 k값은 4

In [36]:
retrieved_docs[0]

Document(id='bb607aa9-23e0-4c3e-86ca-68d8cf4fc2af', metadata={'source': 'tax_docx/소득세법_with_table.docx'}, page_content='제54조의2(공동사업에 대한 소득공제 등 특례) 제51조의3 또는 「조세특례제한법」에 따른 소득공제를 적용하거나 제59조의3에 따른 세액공제를 적용하는 경우 제43조제3항에 따라 소득금액이 주된 공동사업자의 소득금액에 합산과세되는 특수관계인이 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액이 있으면 주된 공동사업자의 소득에 합산과세되는 소득금액의 한도에서 주된 공동사업자가 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액으로 보아 주된 공동사업자의 합산과세되는 종합소득금액 또는 종합소득산출세액을 계산할 때에 소득공제 또는 세액공제를 받을 수 있다. <개정 2012. 1. 1., 2014. 1. 1.>\n\n[전문개정 2009. 12. 31.]\n\n[제목개정 2014. 1. 1.]\n\n\n\n제4절 세액의 계산 <개정 2009. 12. 31.>\n\n\n\n제1관 세율 <개정 2009. 12. 31.>\n\n\n\n제55조(세율) ①거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액(이하 “종합소득산출세액”이라 한다)을 그 세액으로 한다. <개정 2014. 1. 1., 2016. 12. 20., 2017. 12. 19., 2020. 12. 29., 2022. 12. 31.>\n\n\n\n\n\n\n\n\n\n종합소득 과세표준\n\n세율\n\n1,400만원 이하\n\n과세표준의 6퍼센트\n\n1,400만원 초과 5,000만원 이하\n\n84만원 + (1,400만원을 초과하는 금액의 15퍼센트)\n\n5,000만원 초과 8,800만원 이하\n\n624만원 + (5,000만원을 초과하는 금액의 24퍼센트)\n\n8,800만원 초과 1억 5천만원 이하\n\n1,536만원 + (8,800만원을 초과하는 금액의 35퍼센트)\n\n1억5천

In [40]:
retriever = database.as_retriever(search_type='similarity',
                                 search_kwarg={'k':2}
                                 )
retriever_docs = retriever.invoke("연봉이 5천만원인 직장인의 소득세는 얼마인가요?")
retriever_docs[0]

Document(id='bb607aa9-23e0-4c3e-86ca-68d8cf4fc2af', metadata={'source': 'tax_docx/소득세법_with_table.docx'}, page_content='제54조의2(공동사업에 대한 소득공제 등 특례) 제51조의3 또는 「조세특례제한법」에 따른 소득공제를 적용하거나 제59조의3에 따른 세액공제를 적용하는 경우 제43조제3항에 따라 소득금액이 주된 공동사업자의 소득금액에 합산과세되는 특수관계인이 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액이 있으면 주된 공동사업자의 소득에 합산과세되는 소득금액의 한도에서 주된 공동사업자가 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액으로 보아 주된 공동사업자의 합산과세되는 종합소득금액 또는 종합소득산출세액을 계산할 때에 소득공제 또는 세액공제를 받을 수 있다. <개정 2012. 1. 1., 2014. 1. 1.>\n\n[전문개정 2009. 12. 31.]\n\n[제목개정 2014. 1. 1.]\n\n\n\n제4절 세액의 계산 <개정 2009. 12. 31.>\n\n\n\n제1관 세율 <개정 2009. 12. 31.>\n\n\n\n제55조(세율) ①거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액(이하 “종합소득산출세액”이라 한다)을 그 세액으로 한다. <개정 2014. 1. 1., 2016. 12. 20., 2017. 12. 19., 2020. 12. 29., 2022. 12. 31.>\n\n\n\n\n\n\n\n\n\n종합소득 과세표준\n\n세율\n\n1,400만원 이하\n\n과세표준의 6퍼센트\n\n1,400만원 초과 5,000만원 이하\n\n84만원 + (1,400만원을 초과하는 금액의 15퍼센트)\n\n5,000만원 초과 8,800만원 이하\n\n624만원 + (5,000만원을 초과하는 금액의 24퍼센트)\n\n8,800만원 초과 1억 5천만원 이하\n\n1,536만원 + (8,800만원을 초과하는 금액의 35퍼센트)\n\n1억5천

In [45]:
retriever_doc="\n--\n".join([doc.page_content for doc in retriever_docs])
#print(retriever_doc)

# 3. 답변 생성


In [47]:
from openai import OpenAI
client= OpenAI()
query= "연봉이 5천만원인 직장인의 소득세는 얼마인가요?"
response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role':'system','content':'당신은 최고의 한국 소득세법 전문가입니다'},
            {
                'role':'user',
                'content':f"""-[context]를 참고해서 사용자의 질문에 10줄이내로 답변해 주세요.
                -[context]:{retriever_doc}
                -질문 : {query}"""
            }
        ],
    temperature=0.2
)

In [52]:
result=response.choices[0].message

In [53]:
print(result)

ChatCompletionMessage(content='연봉이 5천만원인 직장인의 소득세를 계산하기 위해서는 종합소득세율을 적용해야 합니다. \n\n1. 5천만원의 과세표준에 대해 세율을 적용합니다. \n   - 1,400만원 이하: 6% → 84만원\n   - 1,400만원 초과 5,000만원 이하: 84만원 + (3,600만원 × 15%) = 84만원 + 540만원 = 624만원\n\n2. 따라서, 5천만원의 종합소득에 대한 소득세는 624만원입니다.\n\n결론적으로, 연봉 5천만원인 직장인의 소득세는 624만원입니다.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)


In [57]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
        model='gpt-4.1-mini'
)

In [58]:
from langchain_upstage import ChatUpstage
llm = ChatUpstage(
        model='solar-pro4',
        reasoning_effort='low' #느리지만 더 깊게 추론하여 답변(low,medium,high)
)

In [59]:
prompt=f"""[identity]
-당신은 최고의 한국 소득세법 전문가입니다.
-[context]를 참고하여 사용자의 질문에 답변해주세요. [context]는 다음과 같아요
{retriever_doc}
- 질문 : {query}"""

In [61]:
ai_message = llm.invoke(prompt)
print(ai_message.content)

제공해주신 소득세법 조문을 바탕으로 연봉 5,000만원 직장인의 소득세 계산 구조와 대략적인 금액을 설명드리겠습니다. 다만, **소득세는 총급여액에서 각종 공제를 차감한 `종합소득과세표준`을 기준으로 산출되며, 부양가족, 신용카드 사용액, 연금·보험료 납입, 기타 세액공제 적용 여부 등에 따라 실제 납부세액이 크게 달라지므로** 아래 내용은 조문 기준의 계산 흐름과 가정적 예시임을 먼저 말씀드립니다.

---

### 🔹 소득세 계산의 기본 구조 (제공 조문 기준)
1. **총급여액** → 근로소득공제(제47조 등) 및 기타 소득공제(제50조 기본공제 등) 적용 → **종합소득과세표준** 도출
2. 과세표준에 **제55조 세율** 적용 → **종합소득산출세액**
3. 산출세액에서 **제59조 근로소득세액공제**, 제59조의2 자녀세액공제, 제59조의3 연금계좌세액공제 등 적용 → **최종 납부세액** (지방소득세 10% 별도)

---

### 🔹 제55조(세율) 기준 산출세액 예시
연봉 5,000만원이 **다른 공제 없이 그대로 과세표준**이라고 가정할 경우(비현실적이지만 조문 적용 연습용):

- 5,000만원은 `1,400만원 초과 5,000만원 이하` 과세표준 구간에 해당합니다.
- 제55조 제1항에 따른 산출세액:
  ```
  84만원 + (5,000만원 - 1,400만원) × 15% 
  = 84만원 + 540만원 
  = 624만원
  ```
  ※ 이는 과세표준이 정확히 5,000만원일 때의 산출세액이며, 실제 과세표준은 근로소득공제 등으로 낮아집니다.

---

### 🔹 제59조(근로소득세액공제) 적용 예시
제공된 지문 제59조 제2항에 따르면, 근로소득세액공제는 총급여액 구간별로 상한이 정해져 있습니다.

- 총급여 5,000만원은 `3,300만원 초과 7,000만원 이하` 구간에 해당합니다.
- 동항 제2호 공식 적용:
  ```
  74만원 - [(5,000만원 - 3,300만원) × 8/1,000]
  = 74만원 - (1,700만원

In [62]:
from langchain_upstage import ChatUpstage, UpstageEmbeddings
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv
# 1. 임베딩 & LLM 초기화
load_dotenv()
embedding = UpstageEmbeddings(model = "embedding-query")
llm = ChatUpstage(model = 'solar-pro4')
#2. vector DB load
#pc = Pinecone()
index_name= 'tax-index-table'
vectorStore=PineconeVectorStore(
            embedding=embedding,
            index_name=index_name
                )
#3. Retriever 생성 
# (1) search_type='similarity' : 가장 유사한 top-k개 추출
retriever = vectorStore.as_retriever(
    search_type='similarity',
    search_kwargs={'k':2} # 유사도가 높은 top-2추출
)

#4. 프롬프트템플릿
template= f"""당신은 최고의 한국 소득세법 전문가입니다. 다음 문맥을 참고하여 질몬에 답하세요.
답을 모르면 모른다고 답하세요. 최대 3문장으로 간결하게 답변하세요
질문 : {{query}}
문맥 : {{context}}
답변 : """
prompt = ChatPromptTemplate.from_template(template)
# 5. 검색된 문서를 텍스트로 변환하는 함수
def format_documents(retrieved_docs):
    return '\n--\n'.join([doc.page_content.replace('\n\n',' ') for doc in retrieved_docs])

In [63]:
# 6. RAG 체인 구성(LCEL 방식)
rag_chain = (
    {
        "context":retriever|format_documents,
        "query" : RunnablePassthrough() # 질문만 그대로 전달
    }
    |prompt
    |llm
    |StrOutputParser()
)
rag_chain.invoke('연봉 5천만원인 직장인의 소득세는 얼마인가요?')

'제공된 문맥만으로는 연봉 5천만원인 직장인의 정확한 소득세를 계산할 수 없습니다. 소득세법 제59조(근로소득세액공제)는 총급여 5천만원 구간(한도 66만원)만 규정하고 있으나, 실제 세액을 산출하려면 인적공제, 연금보험료·신용카드 등 기타 소득공제와 자녀·연금계좌 세액공제 적용 여부가 모두 반영되어야 하기 때문입니다. 구체적인 소득공제 및 세액공제 항목을 알려주시면 정확한 계산 기준을 안내해 드릴 수 있습니다.'